In [1]:
import numpy as np
import random
import os
from sklearn.datasets import load_iris
from Benchmarks import get_dataset
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
import matplotlib.pyplot as plt
from sklearn.ensemble import RandomForestClassifier
import scipy.stats as stats
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
import tensorflow as tf
from tensorflow.keras import layers, models
from sklearn.random_projection import GaussianRandomProjection
from sklearn.feature_selection import RFE

In [3]:
# PCA: 降维至累计解释方差超过95%的最小维度

def PCA_RF(seed,data,label):
    np.random.seed=seed
    random.seed=seed
    # Debug #####################################################################################
    # data = load_iris()
    # X = data.data
    # y = data.target
    # (TrainX, TestX, TrainY, TestY) = train_test_split(X, y, test_size=0.2, random_state=seed)
    #############################################################################################
    
    (TrainX, TestX, label1, label2) = train_test_split(data, label, test_size=0.2, random_state=seed)
    TrainY = label1[:, 0]
    TrainR = label1[:, 1:]
    TestY = label2[:, 0]
    TestR = label2[:, 1:]
    
    
    # 对训练集数据进行标准化
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(TrainX)
    X_test_scaled = scaler.transform(TestX)
    
    # 找到合适的维度，这里解释方差大于95%
    pca = PCA() 
    pca.fit(X_train_scaled)
    explained_variance_ratio = pca.explained_variance_ratio_
    cumulative_explained_variance = np.cumsum(explained_variance_ratio)
    num_components = np.argmax(cumulative_explained_variance >= 0.95) + 1
    
    # 降维
    pca = PCA(n_components=num_components)
    TrainX = pca.fit_transform(X_train_scaled)
    TestX = pca.transform(X_test_scaled)
    print(len(TrainX[0]))
    # 创建随机森林分类器
    rf_classifier = RandomForestClassifier(random_state=seed)
    # 训练模型
    rf_classifier.fit(TrainX, TrainY)
    y_hat = rf_classifier.predict(TestX)
    
    # print(accuracy_score(y_hat, TestY))
    acc = 0
    for i in range(len(TestY)):
        if TestY[i] == y_hat[i]:
            acc += 1
        elif stats.ranksums(TestR[i, int(y_hat[i] * 30):int((y_hat[i] + 1) * 30)],
                            TestR[i, int(TestY[i] * 30):int((TestY[i] + 1) * 30)])[1] > 0.05:
            acc += 1

    return acc / len(TestY)

results=np.zeros([4,30])
for i in range(4):
    data, label = get_dataset(i)
    for j in range(1):
        results[i,j]=PCA_RF(j,data,label)
        print(i,j,results[i,j])
        
# np.savetxt('D:\Pythonnnnnn\python\FeatureConstruction\EXP\comparison\FEM\PCA.csv',results,delimiter=',')


17
0 0 0.8752941176470588
20
1 0 0.7678983833718245
20
2 0 0.6186046511627907


D:\Python39\lib\site-packages\numpy\core\fromnumeric.py:86: RuntimeWarning: invalid value encountered in reduce
  return ufunc.reduce(obj, axis, dtype, out, **passkwargs)


18
3 0 0.7738234150136134


In [ ]:
# RP 到30维吧
def RP_RF(seed,data,label):
    np.random.seed=seed
    random.seed=seed
    # Debug #####################################################################################
    # data = load_iris()
    # X = data.data
    # y = data.target
    # (TrainX, TestX, TrainY, TestY) = train_test_split(X, y, test_size=0.2, random_state=seed)
    #############################################################################################
    
    (TrainX, TestX, label1, label2) = train_test_split(data, label, test_size=0.2, random_state=seed)
    TrainY = label1[:, 0]
    TrainR = label1[:, 1:]
    TestY = label2[:, 0]
    TestR = label2[:, 1:]
    
    
    # 对训练集数据进行标准化
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(TrainX)
    X_test_scaled = scaler.transform(TestX)

    # 使用GaussianRandomProjection进行随机映射
    rp = GaussianRandomProjection(n_components=30)
    TrainX = rp.fit_transform(X_train_scaled)
    TestX = rp.transform(X_test_scaled)
    
    # 创建随机森林分类器
    rf_classifier = RandomForestClassifier(random_state=seed)
    # 训练模型
    rf_classifier.fit(TrainX, TrainY)
    y_hat = rf_classifier.predict(TestX)
    
    # print(accuracy_score(y_hat, TestY))
    
    acc = 0
    for i in range(len(TestY)):
        if TestY[i] == y_hat[i]:
            acc += 1
        elif stats.ranksums(TestR[i, int(y_hat[i] * 30):int((y_hat[i] + 1) * 30)],
                            TestR[i, int(TestY[i] * 30):int((TestY[i] + 1) * 30)])[1] > 0.05:
            acc += 1

    return acc / len(TestY)

results=np.zeros([4,30])
for i in range(4):
    data, label = get_dataset(i)
    for j in range(30):
        results[i,j]=RP_RF(j,data,label)
        print(i,j,results[i,j])
        
np.savetxt('D:\Pythonnnnnn\python\FeatureConstruction\EXP\comparison\FEM\RP.csv',results,delimiter=',')

In [58]:
# AE,3个编码层，1瓶颈层，3个解码层，61-128-64-32-32-32-64-128-61 

# 定义自编码器
def build_autoencoder(input_dim):
    # 编码器部分
    inputs = layers.Input(shape=(input_dim,))
    
    # 第一层编码，逐渐减小维度
    x = layers.Dense(128, activation='relu')(inputs)
    x = layers.Dense(64, activation='relu')(x)
    x = layers.Dense(32, activation='relu')(x)
    
    # 潜在空间层（瓶颈层），表示压缩后的数据
    bottleneck = layers.Dense(32, activation='relu')(x)
    
    # 解码器部分，逐渐恢复到原始输入维度
    x = layers.Dense(32, activation='relu')(bottleneck)
    x = layers.Dense(64, activation='relu')(x)
    x = layers.Dense(128, activation='relu')(x)
    
    # 输出层恢复到原始输入维度
    decoded = layers.Dense(input_dim, activation='sigmoid')(x)  # 用sigmoid输出[0, 1]范围的重构数据
    
    # 构建自编码器模型
    autoencoder = models.Model(inputs, decoded)
    
    # 编码器模型，方便提取编码后的潜在空间数据
    encoder = models.Model(inputs, bottleneck)
    
    # 编译自编码器
    autoencoder.compile(optimizer='adam', loss='mse')
    
    return autoencoder, encoder


def AE_RF(seed,data,label):
    np.random.seed=seed
    random.seed=seed
    tf.random.set_seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed) 
    os.environ['TF_DETERMINISTIC_OPS'] = '1'
    
    (TrainX, TestX, label1, label2) = train_test_split(data, label, test_size=0.2, random_state=seed)
    TrainY = label1[:, 0]
    TrainR = label1[:, 1:]
    TestY = label2[:, 0]
    TestR = label2[:, 1:]
    
    # 对训练集数据进行标准化
    scaler = StandardScaler()
    TrainX = scaler.fit_transform(TrainX)
    TestX = scaler.transform(TestX)
    
    autoencoder, encoder = build_autoencoder(len(TrainX[0]))
    
    autoencoder.fit(TrainX, TrainX, epochs=50, batch_size=64, validation_split=0.2)
    
    TrainX = encoder.predict(TrainX)
    TestX = encoder.predict(TestX)

    # 创建随机森林分类器
    rf_classifier = RandomForestClassifier(random_state=seed)
    # 训练模型
    rf_classifier.fit(TrainX, TrainY)
    y_hat = rf_classifier.predict(TestX)
    
    # print(accuracy_score(y_hat, TestY))
    
    acc = 0
    for i in range(len(TestY)):
        if TestY[i] == y_hat[i]:
            acc += 1
        elif stats.ranksums(TestR[i, int(y_hat[i] * 30):int((y_hat[i] + 1) * 30)],
                            TestR[i, int(TestY[i] * 30):int((TestY[i] + 1) * 30)])[1] > 0.05:
            acc += 1

    return acc / len(TestY)

results=np.zeros([4,30])
for i in range(4):
    data, label = get_dataset(i)
    for j in range(30):
        results[i,j]=AE_RF(j,data,label)
        print(i,j,results[i,j])
        
np.savetxt('D:\Pythonnnnnn\python\FeatureConstruction\EXP\comparison\FEM\AE.csv',results,delimiter=',')


Epoch 1/50
22/22 [==============================] - 1s 7ms/step - loss: 1.0828 - val_loss: 0.9177
Epoch 2/50
22/22 [==============================] - 0s 3ms/step - loss: 0.7866 - val_loss: 0.7548
Epoch 3/50
22/22 [==============================] - 0s 3ms/step - loss: 0.7018 - val_loss: 0.7211
Epoch 4/50
22/22 [==============================] - 0s 3ms/step - loss: 0.6832 - val_loss: 0.7115
Epoch 5/50
22/22 [==============================] - 0s 3ms/step - loss: 0.6735 - val_loss: 0.7028
Epoch 6/50
22/22 [==============================] - 0s 3ms/step - loss: 0.6605 - val_loss: 0.6821
Epoch 7/50
22/22 [==============================] - 0s 3ms/step - loss: 0.6465 - val_loss: 0.6761
Epoch 8/50
22/22 [==============================] - 0s 3ms/step - loss: 0.6415 - val_loss: 0.6723
Epoch 9/50
22/22 [==============================] - 0s 3ms/step - loss: 0.6380 - val_loss: 0.6670
Epoch 10/50
22/22 [==============================] - 0s 3ms/step - loss: 0.6348 - val_loss: 0.6662
Epoch 11/50
22/22 [

D:\Python39\lib\site-packages\numpy\core\fromnumeric.py:86: RuntimeWarning: invalid value encountered in reduce
  return ufunc.reduce(obj, axis, dtype, out, **passkwargs)


Epoch 1/50
258/258 [==============================] - 1s 2ms/step - loss: 0.7357 - val_loss: 0.7132
Epoch 2/50
258/258 [==============================] - 0s 2ms/step - loss: 0.6430 - val_loss: 0.7036
Epoch 3/50
258/258 [==============================] - 0s 2ms/step - loss: 0.6366 - val_loss: 0.6982
Epoch 4/50
258/258 [==============================] - 0s 2ms/step - loss: 0.6287 - val_loss: 0.6877
Epoch 5/50
258/258 [==============================] - 0s 2ms/step - loss: 0.6225 - val_loss: 0.6857
Epoch 6/50
258/258 [==============================] - 0s 2ms/step - loss: 0.6224 - val_loss: 0.6795
Epoch 7/50
258/258 [==============================] - 0s 2ms/step - loss: 0.6137 - val_loss: 0.6762
Epoch 8/50
258/258 [==============================] - 0s 2ms/step - loss: 0.6118 - val_loss: 0.6743
Epoch 9/50
258/258 [==============================] - 0s 2ms/step - loss: 0.6103 - val_loss: 0.6728
Epoch 10/50
258/258 [==============================] - 0s 2ms/step - loss: 0.6094 - val_loss: 0.6718

In [6]:
#  LDA
def LDA_RF(seed,data,label):
    np.random.seed=seed
    random.seed=seed
    # Debug #####################################################################################
    # data = load_iris()
    # X = data.data
    # y = data.target
    # (TrainX, TestX, TrainY, TestY) = train_test_split(X, y, test_size=0.2, random_state=seed)
    #############################################################################################
    
    (TrainX, TestX, label1, label2) = train_test_split(data, label, test_size=0.2, random_state=seed)
    TrainY = label1[:, 0]
    TrainR = label1[:, 1:]
    TestY = label2[:, 0]
    TestR = label2[:, 1:]
    
    
    # 对训练集数据进行标准化
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(TrainX)
    X_test_scaled = scaler.transform(TestX)

    # 3. 使用LDA进行降维（保留2个主成分）
    lda = LinearDiscriminantAnalysis(n_components=len(np.unique(TrainY)) - 1)
    TrainX = lda.fit_transform(X_train_scaled, TrainY)
    TestX = lda.transform(X_test_scaled)
    
    print(len(TrainX[0]))
    # 创建随机森林分类器
    rf_classifier = RandomForestClassifier(random_state=seed)
    # 训练模型
    rf_classifier.fit(TrainX, TrainY)
    y_hat = rf_classifier.predict(TestX)
    
    # print(accuracy_score(y_hat, TestY))
    
    acc = 0
    for i in range(len(TestY)):
        if TestY[i] == y_hat[i]:
            acc += 1
        elif stats.ranksums(TestR[i, int(y_hat[i] * 30):int((y_hat[i] + 1) * 30)],
                            TestR[i, int(TestY[i] * 30):int((TestY[i] + 1) * 30)])[1] > 0.05:
            acc += 1

    return acc / len(TestY)

results=np.zeros([4,30])
for i in range(4):
    data, label = get_dataset(i)
    for j in range(30):
        results[i,j]=LDA_RF(j,data,label)
        print(i,j,results[i,j])
        
# np.savetxt('D:\Pythonnnnnn\python\FeatureConstruction\EXP\comparison\FEM\LDA.csv',results,delimiter=',')

7
0 0 0.8235294117647058
7
0 1 0.8211764705882353
7
0 2 0.8258823529411765
7
0 3 0.8564705882352941
7
0 4 0.8235294117647058
7
0 5 0.8588235294117647
7
0 6 0.8305882352941176
7
0 7 0.8705882352941177
7
0 8 0.8235294117647058
7


KeyboardInterrupt: 

In [59]:
# RFE 到30维吧
def RFE_RF(seed,data,label):
    np.random.seed=seed
    random.seed=seed
    # Debug #####################################################################################
    # data = load_iris()
    # X = data.data
    # y = data.target
    # (TrainX, TestX, TrainY, TestY) = train_test_split(X, y, test_size=0.2, random_state=seed)
    #############################################################################################
    
    (TrainX, TestX, label1, label2) = train_test_split(data, label, test_size=0.2, random_state=seed)
    TrainY = label1[:, 0]
    TrainR = label1[:, 1:]
    TestY = label2[:, 0]
    TestR = label2[:, 1:]
    
    # 对训练集数据进行标准化
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(TrainX)
    X_test_scaled = scaler.transform(TestX)

    # 3. 使用一个分类模型（这里使用随机森林）作为评估器
    model = RandomForestClassifier(random_state=seed)
    
    # 4. 创建RFE对象，并进行特征选择
    rfe = RFE(estimator=model, n_features_to_select=30)  # 选择2个最重要的特征
    rfe.fit(X_train_scaled, TrainY)
    
    # 6. 使用选择的特征进行训练和预测
    TrainX = rfe.transform(TrainX)  # 使用选择的特征
    TestX = rfe.transform(TestX)
    
    # 创建随机森林分类器
    rf_classifier = RandomForestClassifier(random_state=seed)
    # 训练模型
    rf_classifier.fit(TrainX, TrainY)
    y_hat = rf_classifier.predict(TestX)
    
    # print(accuracy_score(y_hat, TestY))
    
    acc = 0
    for i in range(len(TestY)):
        if TestY[i] == y_hat[i]:
            acc += 1
        elif stats.ranksums(TestR[i, int(y_hat[i] * 30):int((y_hat[i] + 1) * 30)],
                            TestR[i, int(TestY[i] * 30):int((TestY[i] + 1) * 30)])[1] > 0.05:
            acc += 1

    return acc / len(TestY)

results=np.zeros([4,30])
for i in range(4):
    data, label = get_dataset(i)
    for j in range(30):
        results[i,j]=RP_RF(j,data,label)
        print(i,j,results[i,j])
        
np.savetxt('D:\Pythonnnnnn\python\FeatureConstruction\EXP\comparison\FEM\RFE.csv',results,delimiter=',')

0 0 0.8564705882352941
0 1 0.8211764705882353
0 2 0.8329411764705882
0 3 0.9035294117647059
0 4 0.8352941176470589
0 5 0.8658823529411764
0 6 0.8658823529411764
0 7 0.84
0 8 0.8729411764705882
0 9 0.8329411764705882
0 10 0.8094117647058824
0 11 0.8376470588235294
0 12 0.8258823529411765
0 13 0.8447058823529412
0 14 0.8752941176470588
0 15 0.8611764705882353
0 16 0.84
0 17 0.8541176470588235
0 18 0.8541176470588235
0 19 0.8541176470588235
0 20 0.8447058823529412
0 21 0.851764705882353
0 22 0.8658823529411764
0 23 0.8611764705882353
0 24 0.8376470588235294
0 25 0.8470588235294118
0 26 0.8117647058823529
0 27 0.8658823529411764
0 28 0.8564705882352941
0 29 0.8682352941176471
1 0 0.7725173210161663
1 1 0.7586605080831409
1 2 0.7771362586605081
1 3 0.7829099307159353
1 4 0.7702078521939953
1 5 0.7806004618937644
1 6 0.7794457274826789
1 7 0.7886836027713626
1 8 0.7505773672055427
1 9 0.7598152424942263
1 10 0.7713625866050808
1 11 0.7852193995381063
1 12 0.7644341801385681
1 13 0.7678983833

D:\Python39\lib\site-packages\numpy\core\fromnumeric.py:86: RuntimeWarning: invalid value encountered in reduce
  return ufunc.reduce(obj, axis, dtype, out, **passkwargs)


3 0 0.7633216647218981
3 1 0.7699338778685336
3 2 0.7889926098794243
3 3 0.7605989887203423
3 4 0.7662388175807079
3 5 0.7664332944379619
3 6 0.7683780630105017
3 7 0.7753792298716453
3 8 0.7576818358615325
3 9 0.7631271878646441
3 10 0.7732399844418514
3 11 0.7775184753014391
3 12 0.76390509529366
3 13 0.7728510307273434
3 14 0.7730455075845974
3 15 0.7755737067288992
3 16 0.7681835861532478
3 17 0.7683780630105017
3 18 0.7594321275768183
3 19 0.7578763127187864
3 20 0.7676001555814858
3 21 0.7775184753014391
3 22 0.7559315441462466
3 23 0.7734344612991054
3 24 0.7705173084402956
3 25 0.7689614935822637
3 26 0.7627382341501361
3 27 0.7640995721509141
3 28 0.7640995721509141
3 29 0.7625437572928822
